### Combine runDORAnet.py outputs

Reads `*_molecules_with_pathways.csv` and `*_pathways.txt` written by
`runDORAnet.py`/DORAnet, and produces two outputs:

- **`{jobName}_generateMolecules.csv`** -- molecule-level, one row per compound
  (starters + everything with at least one pathway).
- **`{jobName}_molecules_wPathways.csv`** -- pathway-level, one row per actual
  route found, since DeltaG is fundamentally a reaction/pathway property, not
  a molecule property. A compound reached by 3 different routes gets 3 rows.

All SMILES are canonicalized with RDKit before use. DeltaG values are read
directly from each job's own `*_pathways.txt`; nothing is recomputed.

In [8]:
import ast
from pathlib import Path

import pandas as pd
from rdkit import Chem

### Find job outputs

In [9]:
searchRoot = "."

csvPaths = sorted(Path(searchRoot).rglob("*_molecules_with_pathways.csv"))
print(f"Found {len(csvPaths)} job output(s):")
for p in csvPaths:
    print(f"  {p}")

Found 1 job output(s):
  InosineOnly_molecules_with_pathways.csv


### Helpers: canonicalization and pathway-file parsing

In [10]:
def canonicalSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol) if mol else smiles


def splitPathwayBlocks(pathwaysTxtPath):
    pathwaysTxtPath = Path(pathwaysTxtPath)
    if not pathwaysTxtPath.exists():
        return []
    blocks, current = [], []
    for line in pathwaysTxtPath.read_text(encoding="utf-8").splitlines():
        if line.startswith("pathway number ") and current:
            blocks.append(current)
            current = [line]
        else:
            current.append(line)
    if current:
        blocks.append(current)
    return blocks


def parsePathwayBlockFull(block):
    """Full per-step detail for one pathway block: SMILES, rule names, and
    DeltaG for every step (None where unparseable, e.g. "No_Thermo"), plus
    the final step's product set (the compound(s) this route reaches).
    """
    pathwayNumber = None
    for line in block:
        if line.startswith("pathway number "):
            pathwayNumber = line.replace("pathway number ", "").strip()
            break

    stoichList = []
    for line in block:
        prefix = "reaction SMILES stoichiometry "
        if line.startswith(prefix):
            try:
                stoichList = ast.literal_eval(line[len(prefix):].strip())
            except Exception:
                stoichList = []
            break

    marker = "reaction SMILES, name, and enthalpy:"
    if marker not in block:
        return None
    idx = block.index(marker)
    payload = [x.strip() for x in block[idx + 1:] if x.strip()]

    if stoichList:
        numSteps = len(stoichList)
    elif len(payload) % 3 == 0 and payload:
        numSteps = len(payload) // 3
    else:
        return None

    reactionSmilesList = payload[:numSteps]
    ruleNameList = payload[numSteps:2 * numSteps]
    enthalpyLines = payload[2 * numSteps:3 * numSteps]
    if not reactionSmilesList:
        return None

    stepDeltaGs = []
    for val in enthalpyLines:
        try:
            stepDeltaGs.append(float(val))
        except (ValueError, TypeError):
            stepDeltaGs.append(None)  # e.g. "No_Thermo"

    finalProducts = {canonicalSmiles(s) for s in reactionSmilesList[-1].split(">>")[-1].split(".")}

    return {
        "pathwayNumber": pathwayNumber,
        "numSteps": numSteps,
        "reactionSmilesList": reactionSmilesList,
        "ruleNameList": ruleNameList,
        "stepDeltaGs": stepDeltaGs,
        "finalProducts": finalProducts,
    }


def buildFullPathwaySmiles(reactionSmilesList):
    """Chain consecutive reactions into one merged pathway string, e.g.
    ['A.B>>C.D', 'C.D>>E.F'] -> 'A.B>>C.D>>E.F': the first step's reactants,
    then every step's products in order. Compact and readable, but drops any
    co-reactants/helpers consumed at step 2 onward -- for that complete
    detail, use the ReactionSequence column instead.
    """
    if not reactionSmilesList:
        return ""
    firstReactants = reactionSmilesList[0].split(">>")[0]
    productsPerStep = [rxn.split(">>")[-1] for rxn in reactionSmilesList]
    return ">>".join([firstReactants] + productsPerStep)

### Build both tables, per job

`starterSMILES` is every starter in that job's molecule CSV, semicolon-joined
(DORAnet's pathway files don't record which specific starter fed a given
route when there's more than one). `DeltaG_kcal_per_mol` (worst single step,
DORAnet's own ranking convention) and `TotalDeltaG_kcal_per_mol` (sum of all
steps) are None when any step's value is missing (`AllStepsHaveDeltaG = False`),
rather than being silently computed from a partial set.

`FullPathwaySMILES` is the compact, chained-arrow form (`A.B>>C.D>>E.F`);
`ReactionSequence` keeps the complete, unabridged per-step detail including
every co-reactant/helper -- see buildFullPathwaySmiles's docstring for why
both exist.

In [11]:
moleculeFrames = []
pathwayRows = []
readErrors = []
jobNames = []

for csvPath in csvPaths:
    jobName = csvPath.name.replace("_molecules_with_pathways.csv", "")
    jobNames.append(jobName)
    sourceDir = str(csvPath.parent)

    try:
        jobDf = pd.read_csv(csvPath)
        jobDf["SMILES"] = jobDf["SMILES"].apply(canonicalSmiles)
        jobDf["SourceJob"] = jobName
        moleculeFrames.append(jobDf)

        starterSmiles = ";".join(sorted(jobDf.loc[jobDf["Is_Starter"], "SMILES"]))

        pathwaysTxtPath = csvPath.parent / f"{jobName}_pathways.txt"
        for block in splitPathwayBlocks(pathwaysTxtPath):
            parsed = parsePathwayBlockFull(block)
            if parsed is None:
                continue

            stepDeltaGs = parsed["stepDeltaGs"]
            allStepsHaveDeltaG = all(v is not None for v in stepDeltaGs) and len(stepDeltaGs) > 0
            knownDeltaGs = [v for v in stepDeltaGs if v is not None]
            worstStepDeltaG = max(knownDeltaGs) if allStepsHaveDeltaG else None
            totalDeltaG = sum(knownDeltaGs) if allStepsHaveDeltaG else None
            fullPathwaySmiles = buildFullPathwaySmiles(parsed["reactionSmilesList"])

            for target in sorted(parsed["finalProducts"]):
                pathwayRows.append({
                    "starterSMILES": starterSmiles,
                    "targetSMILES": target,
                    "pathwayLength": parsed["numSteps"],
                    "DeltaG_kcal_per_mol": worstStepDeltaG,
                    "TotalDeltaG_kcal_per_mol": totalDeltaG,
                    "AllStepsHaveDeltaG": allStepsHaveDeltaG,
                    "FullPathwaySMILES": fullPathwaySmiles,
                    "ReactionSequence": " || ".join(parsed["reactionSmilesList"]),
                    "RuleNames": " || ".join(parsed["ruleNameList"]),
                    "PathwayNumber": parsed["pathwayNumber"],
                    "JobName": jobName,
                    "SourceDir": sourceDir,
                })

    except Exception as exc:
        readErrors.append({"csvPath": str(csvPath), "error": str(exc)})

if readErrors:
    print(f"Failed to read {len(readErrors)} file(s):")
    for err in readErrors:
        print(f"  {err['csvPath']}: {err['error']}")

moleculesRaw = pd.concat(moleculeFrames, ignore_index=True) if moleculeFrames else pd.DataFrame()
pathwaysDf = pd.DataFrame(pathwayRows)
print(f"Molecule rows before dedup: {len(moleculesRaw)}")
print(f"Pathway rows: {len(pathwaysDf)}")

Molecule rows before dedup: 4
Pathway rows: 4


In [12]:
pathwaysDf

,starterSMILES,targetSMILES,pathwayLength,DeltaG_kcal_per_mol,TotalDeltaG_kcal_per_mol,AllStepsHaveDeltaG,FullPathwaySMILES,ReactionSequence,RuleNames,PathwayNumber,JobName,SourceDir
0,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CCc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](OC=O)[C@H...,3,None,None,False,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,rule0126_2 || rule0062_19 || rule0028_51,1,InosineOnly,.
1,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CCc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](OC=O)[C@H...,3,None,None,False,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,rule0043_12 || rule0062_19 || rule0028_51,2,InosineOnly,.
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,3,None,None,False,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C...,rule0028_51 || rule0126_2 || rule0062_19,3,InosineOnly,.
3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,3,None,None,False,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C...,rule0028_51 || rule0126_2 || rule0062_19,3,InosineOnly,.


### Deduplicate the molecule table by canonical SMILES

`Is_Starter` is True if any source job used it as a starter, `SourceJobs`
lists every job that reached the compound.

In [13]:
if not moleculesRaw.empty:
    sourceJobsBySmiles = (
        moleculesRaw.groupby("SMILES")["SourceJob"]
        .apply(lambda jobs: ";".join(sorted(set(jobs))))
        .rename("SourceJobs")
    )
    isStarterBySmiles = moleculesRaw.groupby("SMILES")["Is_Starter"].any()

    molecules = (
        moleculesRaw.drop_duplicates(subset="SMILES", keep="first")
        .drop(columns=["Is_Starter", "SourceJob"])
        .merge(isStarterBySmiles, on="SMILES")
        .merge(sourceJobsBySmiles, on="SMILES")
        .sort_values(["Is_Starter", "SMILES"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    molecules = moleculesRaw

print(f"Unique compounds: {len(molecules)}")
print(f"  Starters: {int(molecules['Is_Starter'].sum()) if not molecules.empty else 0}")
print(f"  Generated, with pathways: {int((~molecules['Is_Starter']).sum()) if not molecules.empty else 0}")
molecules.head()

Unique compounds: 4
  Starters: 1
  Generated, with pathways: 3


,SMILES,MolFormula,MolWeight,NumHeavyAtoms,Is_Starter,SourceJobs
0,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,C10H12N4O5,268.0808,19,True,InosineOnly
1,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,C12H14N4O6,310.0913,22,False,InosineOnly
2,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,C21H36N7O16P3S,767.1152,48,False,InosineOnly
3,CCc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](OC=O)[C@H...,C13H16N4O6,324.1070,23,False,InosineOnly


### Save both files

Named after the job if there's exactly one; otherwise `combined_...`, since a
single filename can't cleanly encode multiple job names.

In [14]:
namePrefix = jobNames[0] if len(set(jobNames)) == 1 else "combined"

moleculesCsvName = f"{namePrefix}_generateMolecules.csv"
pathwaysCsvName = f"{namePrefix}_molecules_wPathways.csv"

molecules.to_csv(moleculesCsvName, index=False)
pathwaysDf.to_csv(pathwaysCsvName, index=False)

print(f"Saved: {moleculesCsvName} ({len(molecules)} rows)")
print(f"Saved: {pathwaysCsvName} ({len(pathwaysDf)} rows)")

Saved: InosineOnly_generateMolecules.csv (4 rows)
Saved: InosineOnly_molecules_wPathways.csv (4 rows)
